In [1]:
import os
import glob
import numpy as np
import pandas as pd

# -----------------------
# USER SETTINGS
# -----------------------
data_dir = "./DaVis Data/"   # <-- change this
pattern = "*.csv"                      # or "frame_*.csv"

# Option A (preferred): column names as they appear in your CSV header
# Examples: ["exx", "eyy", "exy"] or ["E_xx", "E_yy", "E_xy"]
strain_cols_by_name = ["Transverse strain Exx - GL [S]", "Axial strain Eyy - GL [S]", "Shear strain Exy - GL [S]"]  # e.g., ["exx", "eyy", "exy"]

# Option B: column indices (0-based) if names are messy/unknown
# e.g., [3,4,5]
strain_cols_by_index = None  # e.g., [3, 4, 5]

# If you want robust range metrics too:
percentiles = (1, 5, 95, 99)

# Optional: drop obvious nonphysical outliers (set None to disable)
# For example: clip_abs = 0.05  # strains beyond ±5% are ignored
clip_abs = None

# Optional: ignore NaNs/infs (recommended True)
drop_nonfinite = True

# -----------------------
# HELPERS
# -----------------------
def pick_strain_columns(df: pd.DataFrame):
    if strain_cols_by_name is not None:
        missing = [c for c in strain_cols_by_name if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns {missing}. Available columns: {list(df.columns)[:20]} ...")
        cols = strain_cols_by_name
    elif strain_cols_by_index is not None:
        cols = [df.columns[i] for i in strain_cols_by_index]
    else:
        raise ValueError("Set either strain_cols_by_name or strain_cols_by_index.")
    return cols

def clean_values(arr: np.ndarray):
    arr = arr.astype(float)
    if drop_nonfinite:
        arr = arr[np.isfinite(arr)]
    if clip_abs is not None:
        arr = arr[np.abs(arr) <= clip_abs]
    return arr

def stats_for_array(arr: np.ndarray):
    arr = clean_values(arr)
    if arr.size == 0:
        return {
            "n": 0, "min": np.nan, "max": np.nan, "mean": np.nan,
            "std": np.nan, "rmse": np.nan, **{f"p{p}": np.nan for p in percentiles}
        }
    out = {
        "n": arr.size,
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0,
        "rmse": float(np.sqrt(np.mean(arr**2))),
    }
    for p in percentiles:
        out[f"p{p}"] = float(np.percentile(arr, p))
    return out

# -----------------------
# LOAD + COMPUTE
# -----------------------
csv_files = sorted(glob.glob(os.path.join(data_dir, pattern)))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir} matching {pattern}")

per_file_rows = []
all_values = {"exx": [], "eyy": [], "exy": []}  # keys are generic; we’ll map below

# Determine column mapping from first file
df0 = pd.read_csv(csv_files[0],sep=";")
cols = pick_strain_columns(df0)
if len(cols) != 3:
    raise ValueError(f"Expected 3 strain columns, got {len(cols)}: {cols}")

comp_names = ["exx", "eyy", "exy"]
col_map = dict(zip(comp_names, cols))
print("Using columns:", col_map)

for f in csv_files:
    df = pd.read_csv(f,sep=";")
    row = {"file": os.path.basename(f)}
    for comp in comp_names:
        c = col_map[comp]
        arr = df[c].to_numpy()
        s = stats_for_array(arr)
        # store per-file stats with component prefix
        for k, v in s.items():
            row[f"{comp}_{k}"] = v
        # accumulate for pooled stats
        all_values[comp].append(clean_values(arr))
    per_file_rows.append(row)

per_file_df = pd.DataFrame(per_file_rows)

# Pooled stats across ALL files / ALL points
pooled_rows = []
for comp in comp_names:
    pooled_arr = np.concatenate(all_values[comp]) if len(all_values[comp]) else np.array([])
    s = stats_for_array(pooled_arr)
    pooled_rows.append({"component": comp, **s})
pooled_df = pd.DataFrame(pooled_rows)

display(per_file_df.head())
display(pooled_df)

# Save summary tables
out1 = os.path.join("./", "dic_error_per_file_summary.csv")
out2 = os.path.join("./", "dic_error_pooled_summary.csv")
per_file_df.to_csv(out1, index=False)
pooled_df.to_csv(out2, index=False)

print("Wrote:")
print(" -", out1)
print(" -", out2)

Using columns: {'exx': 'Transverse strain Exx - GL [S]', 'eyy': 'Axial strain Eyy - GL [S]', 'exy': 'Shear strain Exy - GL [S]'}


,file,exx_n,exx_min,exx_max,exx_mean,exx_std,exx_rmse,exx_p1,exx_p5,exx_p95,...,exy_n,exy_min,exy_max,exy_mean,exy_std,exy_rmse,exy_p1,exy_p5,exy_p95,exy_p99
0,Static Images0002.csv,3190,-0.004984,0.019000,0.005912,0.001430,0.006082,0.002762,0.003771,0.008070,...,3190,-0.009184,0.007406,0.000051,0.000993,0.000994,-0.002328,-0.001460,0.001491,0.002392
1,Static Images0003.csv,3190,-0.000404,0.016702,0.006843,0.001596,0.007027,0.003086,0.004263,0.009320,...,3190,-0.005921,0.005480,0.000122,0.000949,0.000957,-0.002275,-0.001393,0.001579,0.002441
2,Static Images0004.csv,3190,-0.006383,0.009784,0.000492,0.001404,0.001488,-0.002461,-0.001588,0.002754,...,3190,-0.005114,0.006987,-0.000020,0.000925,0.000925,-0.002087,-0.001432,0.001437,0.002216
3,Static Images0005.csv,3189,-0.011616,0.008589,-0.001055,0.002572,0.002780,-0.006599,-0.005120,0.003183,...,3189,-0.009575,0.006424,0.000036,0.001352,0.001352,-0.003212,-0.002052,0.002135,0.003099
4,Static Images0006.csv,3190,-0.010752,0.004791,-0.003323,0.001330,0.003579,-0.006262,-0.005385,-0.001265,...,3190,-0.005184,0.005185,0.000088,0.000854,0.000858,-0.001930,-0.001232,0.001428,0.002218


,component,n,min,max,mean,std,rmse,p1,p5,p95,p99
0,exx,28709,-0.013010,0.022721,0.002390,0.005339,0.005850,-0.006962,-0.005188,0.012134,0.014009
1,eyy,28709,-0.020726,0.019590,-0.001479,0.003591,0.003883,-0.010206,-0.008390,0.003773,0.005415
2,exy,28709,-0.009575,0.007406,0.000089,0.001040,0.001044,-0.002525,-0.001537,0.001706,0.002590


Wrote:
 - ./dic_error_per_file_summary.csv
 - ./dic_error_pooled_summary.csv


In [3]:
df_DICsim = pd.read_csv("Final_DIC_Sim.csv",sep=",")
df_DICsim.head

<bound method NDFrame.head of    run  trans_u1_mean  trans_u1_std  trans_u1_pct20  uniax_u1_mean  \
0    1          -2.28          1.17             0.0          -2.05   
1    2          -2.29          1.12             0.0          -1.87   
2    3          -2.40          1.11             0.0          -1.79   
3    4          -2.02          1.00             0.0          -1.72   
4    5          -2.15          1.12             0.0          -1.90   
5    6          -2.18          1.10             0.0          -1.94   
6    7          -2.07          1.02             0.0          -1.65   
7    8          -1.92          0.94             0.0          -1.37   
8    9          -1.92          0.94             0.0          -1.37   
9   10          -1.96          0.94             0.0          -1.45   

   uniax_u1_std  uniax_u1_pct20  shear_u1_mean  shear_u1_std  shear_u1_pct20  \
0          5.86            2.07          -1.07          9.27            0.92   
1          5.23            1.83        

In [4]:
mean_uniax1_error = np.mean(df_DICsim['uniax_u1_mean'])
std_uniax1_error = np.mean(df_DICsim['uniax_u1_std'])
mean_uniax2_error = np.mean(df_DICsim['uniax_u2_mean'])
std_uniax2_error = np.mean(df_DICsim['uniax_u1_std'])
mean_shear1_error = np.mean(df_DICsim['shear_u1_mean'])
std_shear1_error = np.mean(df_DICsim['shear_u1_std'])
mean_shear2_error = np.mean(df_DICsim['shear_u2_mean'])
std_shear2_error = np.mean(df_DICsim['shear_u2_std'])


In [6]:
print(
f"Simulated DIC errors (mean ± std)\n"
f" Uniax u1: {mean_uniax1_error:.6g} ± {std_uniax1_error:.6g}\n"
f" Uniax u2: {mean_uniax2_error:.6g} ± {std_uniax2_error:.6g}\n"
f" Shear u1: {mean_shear1_error:.6g} ± {std_shear1_error:.6g}\n"
f" Shear u2: {mean_shear2_error:.6g} ± {std_shear2_error:.6g}"
)

DIC_sim_summary = pd.DataFrame({
    "mode": ["uniax_u1", "uniax_u2", "shear_u1", "shear_u2"],
    "mean_error": [mean_uniax1_error, mean_uniax2_error, mean_shear1_error, mean_shear2_error],
    "std_error":  [std_uniax1_error,  std_uniax2_error,  std_shear1_error,  std_shear2_error],
})

print(DIC_sim_summary.to_string(index=False, float_format=lambda x: f"{x:.6g}"))

Simulated DIC errors (mean ± std)
 Uniax u1: -1.711 ± 4.968
 Uniax u2: -3.345 ± 4.968
 Shear u1: -0.82 ± 8.602
 Shear u2: -0.005 ± 8.849
    mode  mean_error  std_error
uniax_u1      -1.711      4.968
uniax_u2      -3.345      4.968
shear_u1       -0.82      8.602
shear_u2      -0.005      8.849
